# Multi-vector search

Interactive front end for `search.py`. Pick a dataset, encoder and ranker, run exhaustive search over the corpus, and inspect the top-k results.

Vectors must already exist (`python src/encode.py --method colbert --datasets <name>`). Use the **aws** kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from encode import DATASETS_DIR, load_texts
from search import run_search
from methods import ChamferRanker, SlicedWassersteinRanker

## Available vector sets

In [ ]:
rows = []
for meta in sorted(DATASETS_DIR.glob("*/vec_*/*/*.meta.json")):
    m = json.loads(meta.read_text())
    rows.append({"dataset": m["dataset"], "split": m["split"], "encoder": m["method"],
                 "tag": meta.name.removesuffix(".meta.json"), "items": m["num_items"],
                 "vectors": m["num_vectors"], "mean vectors/item": m["vectors_per_item"]["mean"]})
pd.DataFrame(rows)

## Configure

- `RANKER`: `"chamfer"` or `"sw"` (sliced Wasserstein).
- `RANKER_OPTS`: Chamfer takes `symmetric`; SW takes `projections`, `quantiles`, `p`, `seed`.
- `QUERY_TAG` / `CORPUS_TAG`: only needed when a dataset has more than one vector set for the encoder.
- `IGNORE_IDENTICAL_IDS`: set `True` for ArguAna, whose queries also appear in the corpus.
- `WRITE`: save the run to `results/` as a `.trec` + `.meta.json`.

In [ ]:
DATASET = "arguana"
ENCODER = "colbert"
RANKER = "chamfer"
RANKER_OPTS = {"symmetric": False}
# RANKER, RANKER_OPTS = "sw", {"projections": 64, "quantiles": 32, "p": 2, "seed": 0}

QUERY_TAG = None
CORPUS_TAG = None
K = 100
IGNORE_IDENTICAL_IDS = DATASET == "arguana"
WRITE = True

## Run

In [ ]:
out = run_search(DATASET, RANKER, ENCODER, QUERY_TAG, CORPUS_TAG, k=K,
                 ignore_identical_ids=IGNORE_IDENTICAL_IDS, write=WRITE, **RANKER_OPTS)
results = out["results"]
out["meta"]

## Inspect results

All hits as a table, then the top documents for one query with their text.

In [ ]:
df = pd.DataFrame([(qid, rank, did, s) for qid, hits in results.items()
                   for rank, (did, s) in enumerate(hits, 1)],
                  columns=["query_id", "rank", "doc_id", "score"])
df.head(10)

In [ ]:
qids, qtexts = load_texts(DATASETS_DIR / DATASET / "queries", "queries")
dids, dtexts = load_texts(DATASETS_DIR / DATASET / "corpus", "corpus")
query_text, doc_text = dict(zip(qids, qtexts)), dict(zip(dids, dtexts))

def show(qid, n=5, width=300):
    print(f"QUERY {qid}\n  {query_text[qid][:width]}\n")
    for rank, (did, s) in enumerate(results[qid][:n], 1):
        print(f"{rank}. {did}  score={s:.4f}\n   {doc_text[did][:width]}\n")

show(qids[0])

## Compare rankers

Runs several configurations without writing files (`write=False`) and lines up their top-1 hits. Rankers can also be passed as instances, which is how to try a new method from `methods/`.

In [ ]:
configs = {
    "chamfer": ChamferRanker(),
    "chamfer-sym": ChamferRanker(symmetric=True),
    "sw": "sw",
}
runs = {name: run_search(DATASET, r, ENCODER, QUERY_TAG, CORPUS_TAG, k=10,
                         ignore_identical_ids=IGNORE_IDENTICAL_IDS, write=False)["results"]
        for name, r in configs.items()}
pd.DataFrame({name: {q: hits[0][0] for q, hits in res.items()} for name, res in runs.items()}).head(10)